# Earnalism narration job `narration-film-01`

**Prepared, not run.** Input zip SHA256 `2b427306d2b9b2c62bfd168f4013ab4f6d84094d04d3b6364dfc4e294e37cad4`. Copy `inputs.zip` to `/content/drive/MyDrive/earnalism-audiovisual/bicharak/narration-film-01/` in Drive, set `RUN = True` in the first cell, then run all. Outputs and `manifest.json` (sha256 of every output) land in the same folder.


In [ ]:
RUN = False

In [ ]:
if RUN is not True:
    raise RuntimeError("RUN is False; set RUN = True in the first cell to run this job")
import pathlib
JOB_ID = 'narration-film-01'
TITLE = 'bicharak'
KIND = 'narration'
INPUT_ZIP_SHA256 = '2b427306d2b9b2c62bfd168f4013ab4f6d84094d04d3b6364dfc4e294e37cad4'
MODEL_REPO = 'ai4bharat/indic-parler-tts'
MODEL_REVISION = '7b527af5ee8ed1f9a28d80b19703ed9bb8ba10ca'
STEPS = 1
DRIVE_JOB_DIR = '/content/drive/MyDrive/earnalism-audiovisual/bicharak/narration-film-01'


In [ ]:
if RUN is not True:
    raise RuntimeError("RUN is False; set RUN = True in the first cell to run this job")
import hashlib, json, pathlib, zipfile
from google.colab import drive
drive.mount("/content/drive")
OUT_DIR = pathlib.Path(DRIVE_JOB_DIR)
INPUT_ZIP = OUT_DIR / "inputs.zip"
if not INPUT_ZIP.is_file():
    raise RuntimeError("Copy inputs.zip into " + str(OUT_DIR) + " first")
actual = hashlib.sha256(INPUT_ZIP.read_bytes()).hexdigest()
if actual != INPUT_ZIP_SHA256:
    raise RuntimeError("inputs.zip sha256 mismatch: " + actual)
with zipfile.ZipFile(INPUT_ZIP) as z:
    JOB = json.loads(z.read("job.json"))
if JOB["job_id"] != JOB_ID:
    raise RuntimeError("job.json belongs to a different job")
print("inputs verified for", JOB_ID)


In [ ]:
# Narration: one DRY WAV per voice unit with Indic Parler-TTS (revision pinned at build time).
# No pause is written here: eav.film.narration_track inserts every pause exactly once (R8).
if RUN is not True:
    raise RuntimeError("RUN is False; set RUN = True in the first cell to run this job")
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "git+https://github.com/huggingface/parler-tts.git", "soundfile"], check=True)
import soundfile as sf, torch
from parler_tts import ParlerTTSForConditionalGeneration
from transformers import AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
model = ParlerTTSForConditionalGeneration.from_pretrained(
    MODEL_REPO, revision=MODEL_REVISION).to(device)
tokenizer = AutoTokenizer.from_pretrained(MODEL_REPO, revision=MODEL_REVISION)
desc_tokenizer = AutoTokenizer.from_pretrained(
    model.config.text_encoder._name_or_path)
GENERATION = dict(JOB.get("generation", {}))  # pinned roster preset settings (R9)
if GENERATION and "do_sample" not in GENERATION:
    GENERATION["do_sample"] = True  # a temperature only applies when sampling
MODEL_INFO = {"repo": MODEL_REPO, "revision": MODEL_REVISION, "generation": GENERATION}

rate = model.config.sampling_rate
for unit in JOB["units"]:
    d = desc_tokenizer(unit["description"], return_tensors="pt").to(device)
    p = tokenizer(unit["text"], return_tensors="pt").to(device)
    audio = model.generate(input_ids=d.input_ids, attention_mask=d.attention_mask,
                           prompt_input_ids=p.input_ids,
                           prompt_attention_mask=p.attention_mask, **GENERATION)
    wav = audio.cpu().numpy().squeeze().astype("float32")
    sf.write(str(OUT_DIR / (unit["unit_key"] + ".wav")), wav, rate)
    print("wrote", unit["unit_key"])


In [ ]:
if RUN is not True:
    raise RuntimeError("RUN is False; set RUN = True in the first cell to run this job")
import hashlib, json
outputs = {}
for p in sorted(OUT_DIR.iterdir()):
    if p.is_file() and p.name not in ("manifest.json", "inputs.zip"):
        outputs[p.name] = hashlib.sha256(p.read_bytes()).hexdigest()
if not outputs:
    raise RuntimeError("no outputs were produced")
manifest = {"schema": "earnalism.audiovisual-outputs.v1", "job_id": JOB_ID, "title": TITLE,
            "kind": KIND, "input_zip_sha256": INPUT_ZIP_SHA256, "model": MODEL_INFO,
            "outputs": outputs}
(OUT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2, sort_keys=True) + "\n")
print("manifest.json written:", len(outputs), "files")
